<a href="https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/Cap_and_Pile_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cap and pile design — portable Colab edition

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/Cap_and_Pile_Design.ipynb)

Upload this notebook through **Colab → File → Upload notebook**, then **Runtime → Run all**. The calculation package is included in this file. It uses the reviewed update without needing a GitHub publication.

Start with your solved FBMP XML. The reference trial CSV supplied beside this notebook belongs to a separate concrete end-bent run and is not loaded automatically.


**Reinforcement inputs:** pile bottom bars are continuous through the full cap. **Between-pile row counts are ADDITIONAL bars**, not total span bars. Combined span steel = continuous + added. Zero adds no bars. The live summary reports both counts and areas. Older saved total-count cases are converted once and display a conversion notice; review mixed bar sizes.

The live cage includes minimum clear-spacing checks, layer alignment, hook fit/clash checks, a bottom-steel plan, reinforcement elevation with actual pile embedment, and separate hoop pitch/clear-gap/leg-spacing views. Enter and confirm the mix aggregate size. Standard hook geometry does not establish development or cutoff adequacy; end anchorage and pile-head hoop detailing remain pending.

**LRFD regions & checks:** Enable the actual transverse schedule, then open this tab to set the load path, pile connections, coating, splices and closure details. The calculations use AASHTO LRFD 10th edition (2024) and FDOT SDG 2026. Unknown regions retain full longitudinal interaction. The tab and exported report show region decisions, development lengths, actual intersected stirrup legs, shear/torsion, face reinforcement and numerical working. Pending detail inputs are not treated as passed checks.


# Cap and pile design
**Signed axial / LRFD update:** Restart the runtime and run all cells. Load the saved notebook/case, then reimport fixed-depth cap XML when axial provenance is unresolved. The minimum-tip source remains separate. Open LRFD regions & checks for concurrent station, signed axial, cracking, strain and theta traces. See SIGNED_AXIAL_LRFD.md in the update package.
One notebook with **two independent analysis inputs** for pier and end-bent caps supported on a single pile row.

1. **Run all.** In **Cell 1 · Minimum-tip study and geotech handoff**, upload the **Minimum-tip XML**. This model supplies the pile profiles and the original factored pile-head compression/uplift for geotech.
2. Open **Minimum tip**, enter pile cutoff and design ground/scour elevations, then paste the five trial columns or load the trial CSV. Calculate the minimum tip. Trial history remains separate from the solved XML. The default trial allowance is **Lcrit + 5 ft**; the profile comparison and its allowance remain explicit.
3. Open **Geotech handoff**. **Reaction margin (%)** defaults to **5%** and is editable (0 uses the original reactions). Handoff demand = original factored pile-head demand × (1 + margin/100). Original values, increased values, governing records and source are retained. This is an internal project margin, additional to the FBMP load factors, applied to compression and uplift magnitudes only.
4. In **Cell 2 · Fixed-depth cap design**, upload the **Fixed-depth cap XML**, review the preview and click **Apply XML inputs**. Design the cap with those forces. Loading this XML preserves the minimum-tip study, and the handoff margin does not change cap forces or pile profiles.
5. Use **Save both sections** to download **notebook_state.json**. **Load notebook / case** restores both analyses, trials, elevations, margin and cap inputs from that file. Existing **selected_case.json** and **pile_review.json** files remain available for separate cap and pile restores. The full cap review bundle also includes the combined state when a pile study is loaded.

Under **Geometry → Pile head**, confirm embedment and clear gap. Set reinforcement under **Steel**, inspect the cage, force diagrams and checks, then search practical layouts. **Cap section and steel study** compares trial dimensions using the selected force basis. Rerunning Cell 1 preserves the loaded inputs and reconnects the displayed cap section; rerunning Cell 2 only redisplays the cap controls.

No FBMP solver runs here. A changed minimum tip, cap geometry or stiffness requires review and, where needed, a fresh FBMP analysis and import. These cells keep source roles explicit; they do not establish that the two models use matching assumptions or automatically verify the final pile-tip elevation. Completed searches and section studies need a new run after loading saved inputs.

**Calculation report:** Generate a fresh HTML report in the cap export panel for equations, evaluated results and interactive plots. Pile review and geotech handoff remain separate reports in the full bundle. **Force diagrams** compares the analyzed demands with current sectional resistances. **Dimensions** shows current geometry, while force diagrams retain the analyzed geometry. Backwall/wingwall design, earth-pressure load generation, footing/column caps and strut-and-tie design remain outside this calculation.


In [ ]:
# @title Disable output scrolling
from IPython.display import Javascript, display
from IPython import get_ipython

# Expand output height before each cell runs.
def disable_colab_scrolling(*_):
    display(Javascript('google.colab.output.setIframeHeight(0, true, {maxHeight: 10000})'))

# Keep one callback when this cell or Run all is executed again.
_scroll_events = get_ipython().events
for _callback in list(_scroll_events.callbacks.get('pre_run_cell', [])):
    if getattr(_callback, '__name__', '') == 'disable_colab_scrolling':
        _scroll_events.unregister('pre_run_cell', _callback)
_scroll_events.register('pre_run_cell', disable_colab_scrolling)
print("Output scrolling disabled for future cell executions.")


In [ ]:
# @title Start the portable tool
from pathlib import Path
import base64, hashlib, importlib, io, os, subprocess, sys, zipfile
try:
    from google.colab import output as colab_output
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
REPO_REF = "portable package"
revision = "52db2c3b3edb5c899ca6edf4a4cbc5820da95b9c849121f301cc17e75a52ec3b"
PACKAGE = "..."
_package_bytes = base64.b64decode(PACKAGE)
if hashlib.sha256(_package_bytes).hexdigest() != revision:
    raise RuntimeError("Embedded calculation package is damaged.")
ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / ("cap-pile-tool-" + revision[:12])
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_package_bytes)) as _package:
    for _name in _package.namelist():
        _target = (ROOT / _name).resolve()
        if not _target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Invalid embedded package path.")
        _contents = _package.read(_name)
        if _target.exists() and _target.read_bytes() != _contents:
            raise RuntimeError("An extracted calculation file was edited. Preserve your edits and exports before using a fresh notebook folder.")
        _target.parent.mkdir(parents=True, exist_ok=True)
        if not _target.exists():
            _target.write_bytes(_contents)
if IN_COLAB:
    requirements = ROOT / "requirements-pier-cap-colab.txt"
    if not (ROOT / "pier_cap").is_dir() or not requirements.is_file():
        raise RuntimeError("The embedded package is incomplete. Reopen the delivered notebook.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(requirements)], check=True)
    # Pip does not replace classes already in memory. Require Colab's
    # tested widget protocol and a session matching the installed files.
    import ipywidgets as widget_runtime
    from importlib.metadata import version as installed_version
    loaded_widgets = widget_runtime.__version__
    installed_widgets = installed_version("ipywidgets")
    if loaded_widgets != installed_widgets or loaded_widgets != "7.7.1":
        raise RuntimeError(
            f"Widget session restart required: Python loaded ipywidgets {loaded_widgets}, "
            f"but {installed_widgets} is installed. Use Runtime → Restart session, "
            "then reload this browser page and Run all. Do not delete the runtime. Save any existing case before "
            "restarting. Upload/search controls will not be created in this mixed state."
        )
    import plotly as plot_runtime
    if plot_runtime.__version__ != "5.24.1" or installed_version("plotly") != "5.24.1":
        raise RuntimeError(
            "Chart session restart required. Save your case, use Runtime → Restart session, "
            "reload this browser page, then Run all. Colab needs Plotly 5.24.1 for live chart widgets."
        )
    # Colab may import widgets before its kernel exists; register handlers now.
    widget_runtime.register_comm_target(get_ipython().kernel)
    colab_output.enable_custom_widget_manager()
    os.chdir(ROOT)
    # A previous run (even a failed import) may have cached the old package.
    for module_name in list(sys.modules):
        if module_name == "pier_cap" or module_name.startswith("pier_cap."):
            del sys.modules[module_name]
    importlib.invalidate_caches()
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

# Explain a partially published update before importing the notebook API.
optimizer_module = importlib.import_module("pier_cap.optimizer")
required_search_api = ("SearchConfig", "search", "candidate_case", "filter_candidates", "candidate_dc", "governing_check")
missing_search_api = [name for name in required_search_api if not hasattr(optimizer_module, name)]
if missing_search_api:
    raise RuntimeError(
        "The notebook and supporting calculation files are different versions. "
        f"Missing: {', '.join(missing_search_api)}. Push the notebook and the entire "
        "pier_cap/ folder together, then reopen the notebook and Run all."
    )

from IPython.display import display, HTML
from pier_cap.model import default_case, evaluate, set_inputs, formula_trace, DEFINITIONS
try:
    from pier_cap.fbmp import import_fbmp_xml
    from pier_cap.widgets import CapNotebook
except ImportError as exc:
    raise RuntimeError("The XML importer files are missing or out of date. Push this notebook and the entire pier_cap/ folder together, then reopen and Run all.") from exc
from pier_cap.optimizer import SearchConfig, search, candidate_case, filter_candidates, candidate_dc, governing_check
from pier_cap.io import load_case, write_case, import_workbooks, export_bundle, export_blockpad
try:
    from pier_cap.sections import SectionGrid, CostRates, run_section_study, section_rows
    from pier_cap.section_widgets import SectionStudy
except ImportError as exc:
    raise RuntimeError("The section-study files are missing or out of date. Push this notebook and the entire pier_cap/ folder together, then reopen from GitHub and Run all.") from exc

if IN_COLAB:
    print(f"Colab ready · {REPO_REF} · supporting calculation revision {revision}")

case = default_case()
print("Ready. Upload an XML or load a saved case in the workbench below.")


In [ ]:
# @title Cell 1 · Minimum-tip study and geotech handoff
from pier_cap.widgets import CapNotebook
from pier_cap.section_widgets import rebind_study
# Preserve imported inputs if this cell (or Run all) is run again in this runtime.
from copy import deepcopy
_previous_app = globals().get("app")
_active_case = deepcopy(_previous_app.case) if _previous_app is not None else case
_previous_receipt = deepcopy(getattr(_previous_app, "import_receipt", None))
_previous_json_receipt = deepcopy(getattr(getattr(_previous_app, "case_import", None), "receipt", None))
_previous_pile_panel = getattr(_previous_app, "pile_review", None)
_previous_pile_state = _previous_pile_panel.snapshot() if _previous_pile_panel is not None and _previous_pile_panel.review is not None else None
app = CapNotebook(_active_case, export_root=ROOT / "exports")
if _previous_pile_state is not None:
    app.pile_review.restore(_previous_pile_state)
if _previous_app is not None:
    for _name, _control in _previous_app.search_lists.items():
        if _name == "span_counts" and _active_case.get("schema_version", 1) < 3:
            continue  # Old span search counts meant total steel, not added steel.
        if _name in app.search_lists:
            if _name == "skin_counts" and 0 not in _control.options:
                # Upgrade the old search, which could not select 0–3 bars per side.
                if tuple(_control.value) != (5, 6, 7):
                    app.search_lists[_name].value = tuple(sorted({0, *_control.value}))
            else:
                app.search_lists[_name].value = _control.value
    if "bottom_counts" in _previous_app.search_lists:
        for _name in ("pile_counts",):
            app.search_lists[_name].value = _previous_app.search_lists["bottom_counts"].value
        for _name in ("pile_bars", "span_bars"):
            app.search_lists[_name].value = _previous_app.search_lists["main_bars"].value
    for _name in ("limit", "dc_limit", "dc_scope", "objective", "page_size"):
        getattr(app, _name).value = getattr(_previous_app, _name).value
if _previous_receipt is not None:
    app.import_receipt = _previous_receipt
    app.refresh()
# A previously displayed section study must follow this new workbench instance.
_existing_study = globals().get("section_app")
if _existing_study is not None:
    if hasattr(_existing_study, "rebind"):
        section_app = rebind_study(_existing_study, app)
    else:
        _existing_study.close()
        section_app = None
        print("Rerun the section-study cell to connect it to these inputs.")
if _previous_app is not None:
    if hasattr(_previous_app, "close"):
        _previous_app.close()
    else:
        for _figure in _previous_app.figures:
            _figure.close()
        _previous_app.ui.close()
if _previous_json_receipt is not None:
    app.case_import.restore_receipt(_previous_json_receipt)
_previous_force_panel = getattr(_previous_app, "force_diagrams", None)
_previous_overlay = getattr(_previous_force_panel, "show_resistance", None)
if _previous_overlay is not None:
    app.force_diagrams.show_resistance.value = _previous_overlay.value
# Keep an already displayed Cell 2 connected when Cell 1 is rerun.
_previous_cap_ui = getattr(_previous_app, "cap_ui", None)
if _previous_cap_ui is not None and _previous_cap_ui.comm is not None:
    _previous_cap_ui.children = app.cap_ui.children
    app.cap_ui.close()
    app.cap_ui = _previous_cap_ui
app.display("minimum_tip");

In [ ]:
# @title Cell 2 · Fixed-depth cap design
if "app" not in globals():
    print("Run Cell 1 first to initialize the notebook, then run this cell.")
else:
    app.display("cap");


In [ ]:
# @title Cap section and steel study
from pier_cap.section_widgets import rebuild_study
section_app = rebuild_study(globals().get("section_app"), app)
section_app.display();
